# 01 — Global ocean surface heat fluxes

Global snapshot maps of the ocean-side surface heat flux terms on the LLC2160 grid:

- $Q_{net}$ — net surface heat flux including shortwave (`oceQnet`),
- $Q_{sw}$ — net shortwave radiation (`oceQsw`),
- $Q_{ns} = Q_{net} - Q_{sw}$ — non-solar flux (latent + sensible + net longwave).

Per `mit/readme.txt`, this dataset stores `oceQnet`/`oceQsw` positive **upward**
(">0 decreases theta"); `to_positive_down` reads that from the attached metadata and
flips both to the analysis convention (positive down = ocean warming). Global maps
use area-weighted binning of the ~2–4 km cells onto a 0.25° lat-lon grid; selecting one
snapshot reads exactly one ~243 MB file per variable.

In [ ]:
# Environment check: run on SciServer (Kraken domain, with the Poseidon DYAMOND
# ceph volume attached), or set DYAMOND_ROOT to a local subset.
# SciServer containers do not persist `pip install --user` across restarts, so
# fall back to importing directly from the repo's src/ tree if needed.
try:
    from dyamond_fluxes import dyamond_root
except ModuleNotFoundError:
    import sys
    from pathlib import Path as _P

    sys.path.insert(0, str((_P.cwd() / ".." / "src").resolve()))
    from dyamond_fluxes import dyamond_root

root = dyamond_root()
print(f"DYAMOND root: {root}")

In [ ]:
# Local dask cluster: parallel file reads with bounded memory. Adjust n_workers
# to the container allocation (check `free -h`).
from dask.distributed import Client, LocalCluster

cluster = LocalCluster(n_workers=4, threads_per_worker=2, memory_limit="8GB")
client = Client(cluster)
client

In [ ]:
from dyamond_fluxes import open_ocean_dataset

ds = open_ocean_dataset(["oceQnet", "oceQsw"])
ds

In [ ]:
# Choose a snapshot: boreal summer example. Change freely.
SNAPSHOT = "2020-07-15T12:00"
snap = ds.sel(time=SNAPSHOT, method="nearest")
print("snapshot:", snap.time.values)

In [ ]:
from dyamond_fluxes import nonsolar_flux, to_positive_down

qnet = to_positive_down(snap["oceQnet"])
qsw = to_positive_down(snap["oceQsw"])
qns = nonsolar_flux(qnet, qsw)

# Land mask: Depth == 0 over land in the MITgcm grid files.
ocean = ds["Depth"] > 0
qnet, qsw, qns = (q.where(ocean) for q in (qnet, qsw, qns))

In [ ]:
from pathlib import Path

from dyamond_fluxes.plotting import plot_global

FIGDIR = Path("../figures")
FIGDIR.mkdir(exist_ok=True)

lon, lat, area = ds["XC"], ds["YC"], ds["rA"]

fig, ax, qnet_binned = plot_global(
    qnet.load(), lon, lat, area=area,
    title=f"Net surface heat flux, {str(snap.time.values)[:16]}",
)
fig.savefig(FIGDIR / "qnet_global.png", dpi=200, bbox_inches="tight")

In [ ]:
fig, ax, _ = plot_global(
    qsw.load(), lon, lat, area=area, diverging=False,
    title=f"Net shortwave radiation, {str(snap.time.values)[:16]}",
)
fig.savefig(FIGDIR / "qsw_global.png", dpi=200, bbox_inches="tight")

In [ ]:
fig, ax, _ = plot_global(
    qns.load(), lon, lat, area=area,
    title=f"Non-solar heat flux (latent + sensible + net LW), {str(snap.time.values)[:16]}",
)
fig.savefig(FIGDIR / "qns_global.png", dpi=200, bbox_inches="tight")

In [ ]:
from dyamond_fluxes import area_weighted_mean

for name, q in [("Qnet", qnet), ("Qsw", qsw), ("Qns", qns)]:
    value = float(area_weighted_mean(q, area.where(ocean)).compute())
    print(f"global ocean-mean {name}: {value:8.2f} W m-2")

The instantaneous global-mean $Q_{net}$ reflects the diurnal/seasonal phase of the
snapshot, not the climatological imbalance. A time-mean over the record is a dask
reduction over `time` — budget roughly (number of dumps × 243 MB) of reads and run it on
a subsampled time series first (e.g. `ds.isel(time=slice(None, None, 24))` for daily).